# ShopDesk, Section 4 Lab 2: Hooks for Normalization and Compliance

A beginner-friendly notebook that uses **hooks** to clean up tool output and to enforce
compliance. A `PostToolUse` hook appends a **normalized** view of a raw payload (Unix time to
ISO, status code to a word), and a `PreToolUse` hook enforces a **refund threshold**. Built on
the **Claude Agent SDK**, running **Sonnet** (`claude-sonnet-4-6`) through your **Anthropic API
key**.

## The real-world scenario

Tools return whatever the backend hands over: a Unix timestamp, a numeric status code. If the
model reasons over that raw shape it makes small, avoidable mistakes. And some actions, like a
large refund, should never fire without a check. Both problems are solved at the **hook** layer,
where you can reshape output and stop invalid actions deterministically.

The question this lab answers: **how do you normalize tool output before the model uses it, and
how do you block an action that breaks a compliance rule?**

## Objectives

- Write a `PostToolUse` hook that **normalizes** a raw tool payload and appends the clean view.
- Intercept tool calls with a `PreToolUse` hook to **block invalid actions**.
- Add a **compliance rule**: deny a refund over a threshold amount so a human can approve it.

## What you'll observe

- The pure-Python `normalize()` turns a Unix timestamp into ISO time and a status code into a
  word.
- Live, the `PostToolUse` hook appends that normalized view after the raw tool result.
- A small refund passes the threshold check; a large one is blocked by the compliance gate.

## How to run

Run top to bottom. The `normalize()` and compliance-check cells are pure Python and run
anywhere. The hook-wired cells call Claude, so paste a real key into **Setup 2/3** and re-run
from the top; otherwise they skip. **Node.js 18+** must be installed for the Agent SDK.

## 0. Setup

**This cell:** installs the packages. The **Agent SDK** provides the tools and the hooks
API; the base SDK and dotenv handle the key. The Agent SDK also needs Node.js 18+, which cannot
be pip-installed.

In [ ]:
# ===== SETUP 1/3 - install the Agent SDK =====
%pip install -q claude-agent-sdk anthropic python-dotenv

**This cell:** imports, the model, the `RUN_LIVE` switch, and `run_async()` so the async
hooked runs can be called like ordinary functions.

In [ ]:
# ===== SETUP 2/3 - imports, the model, the switch, and an async runner =====
import os                                       # read the API key from the environment
import sys                                      # detect Windows (it needs a special event loop)
import json                                     # parse and build tool payloads
import asyncio                                  # the Agent SDK is async; we drive it ourselves
import threading                                # run that async loop in a side thread (notebook-safe)
from datetime import datetime, timezone         # for the timestamp conversion in normalize()

try:                                            # load a .env file if present
    from dotenv import load_dotenv              #   import the loader
    load_dotenv()                               #   read .env into environment variables
except Exception:                               # not installed? that is fine
    pass                                        #   set the key another way

MODEL = "claude-sonnet-4-6"                      # the Sonnet model the agent will use

os.environ.setdefault("ANTHROPIC_API_KEY", "sk-ant-...")     # placeholder unless you set a real key
_key = os.environ["ANTHROPIC_API_KEY"]           # read whatever key is set
RUN_LIVE = _key.startswith("sk-ant-") and _key != "sk-ant-..."   # True only for a real key

def run_async(make_coro):                        # run any async Agent SDK call, notebook-safe
    box = {}                                     #   carries the result/error out of the thread
    def worker():                                #   runs in its own thread
        loop = asyncio.ProactorEventLoop() if sys.platform == "win32" else asyncio.new_event_loop()
        asyncio.set_event_loop(loop)             #     make it this thread's loop
        try:    box["value"] = loop.run_until_complete(make_coro())   # run to completion
        except Exception as e: box["error"] = e  #     capture any error
        finally: loop.close()                    #     always close the loop
    t = threading.Thread(target=worker); t.start(); t.join()   # run it and wait
    if "error" in box: raise box["error"]        #   surface any error here
    return box.get("value")                      #   hand back the result

print("live model calls:", "ON" if RUN_LIVE else "OFF (using a placeholder key)")

**This cell:** the shared **ShopDesk** bits this lab needs: the status-code map used by
`normalize()`, and the refund threshold the compliance gate enforces.

In [ ]:
# ===== SETUP 3/3 - the status map and the compliance threshold =====
STATUS_NAMES = {1: "processing", 2: "shipped", 3: "delivered"}   # status code -> readable word
REFUND_LIMIT = 500.0                             # refunds over this need a human (the compliance rule)
print("status map ready | refund limit: $", REFUND_LIMIT)

**This cell:** the **narrator** `stream_run()`. It runs one `query()` and prints each tool
call and the final text; the hooks print when they fire, so you can see normalization and the
compliance block happen.

In [ ]:
# ===== stream one hooked run and narrate it =====
from claude_agent_sdk import (                     # the Agent SDK pieces we use:
    query, ClaudeAgentOptions,                     #   run + options
    tool, create_sdk_mcp_server, HookMatcher,      #   define tools, bundle them, wire hooks
    AssistantMessage, ResultMessage, TextBlock, ToolUseBlock,   # message + block types
)

async def stream_run(options, prompt):             # run query() and print what happens
    print("USER:", prompt)                         #   echo the request
    answer = ""                                    #   keep the final text
    async for message in query(prompt=prompt, options=options):   # stream every message
        if isinstance(message, AssistantMessage):  #     the model spoke
            for block in message.content:          #       walk its blocks
                if isinstance(block, ToolUseBlock):#       a tool call...
                    print("  -> tool:", block.name.split("__")[-1], block.input)
                elif isinstance(block, TextBlock): #       ...or text
                    answer = block.text            #         remember the latest text
        elif isinstance(message, ResultMessage):    #     the run finished
            pass
    print("ANSWER:", answer)                        #   the final answer
    return answer

---

### 🎯 Lab objective - clean the output, enforce the rule

**What you build:** a `normalize()` transform, a `PostToolUse` hook that appends its output, and a
`PreToolUse` compliance gate that blocks over-limit refunds.

**Why it helps you build real solutions:** normalizing at the hook layer means every later step
reasons over clean, consistent data, and a compliance gate in code stops a costly action before it
runs, not after.

**How you'll see it:** the hook prints a clean companion payload, and the compliance gate denies a
refund over the limit while letting a small one through.

**This cell:** the **`normalize()`** transform, in pure Python. It turns a Unix timestamp
into an ISO string and a numeric status into a word, leaving everything else untouched. This is
the logic the hook will apply; here it runs standalone so you can see it clearly.

In [ ]:
# ===== normalize a raw payload: Unix -> ISO, code -> word =====
def normalize(raw):                                # raw tool payload -> a clean payload
    out = dict(raw)                                #   copy so we never mutate the input
    if "event_ts" in out:                          #   a Unix timestamp present?
        out["event_time"] = datetime.fromtimestamp(#     -> an ISO 8601 string
            out.pop("event_ts"), timezone.utc).isoformat()
    if "status" in out and out["status"] in STATUS_NAMES:   # a numeric status present?
        out["status"] = STATUS_NAMES[out["status"]]         #   -> a readable word
    return out                                     #   the cleaned payload

print("raw:  ", {"order_id": "A1", "event_ts": 1718000000, "status": 2})   # what the tool returns
print("clean:", normalize({"order_id": "A1", "event_ts": 1718000000, "status": 2}))  # what we want

**This cell:** the **compliance rule** in pure Python: a refund at or under the limit is
allowed, anything over needs a human. Testing it as a plain function first makes the hook version
easy to trust. It runs offline.

In [ ]:
# ===== the compliance rule: a refund threshold =====
def check_compliance(amount):                      # amount -> (allowed?, reason)
    if amount > REFUND_LIMIT:                       #   over the limit?
        return False, f"refund ${amount} over ${REFUND_LIMIT} limit -> needs human approval"
    return True, "within limit"                     #   otherwise fine

for amt in [50, 500, 900]:                          # try a few amounts
    print(f"${amt:>4}:", check_compliance(amt))     #   50 ok, 500 ok, 900 blocked

**This cell:** the **Agent SDK tools**. `get_event` returns a deliberately *raw* payload (a
Unix timestamp and a status code) so the normalization hook has something to clean.
`process_refund` takes an amount so the compliance gate has something to check.

In [ ]:
# ===== the SDK tools =====
@tool("get_event", "Get the latest event for an order (raw fields).", {"order_id": str})
async def sdk_get_event(args):                     # returns a raw payload on purpose
    raw = {"order_id": args["order_id"], "event_ts": 1718000000, "status": 2}   # Unix + code
    return {"content": [{"type": "text", "text": json.dumps(raw)}]}

@tool("process_refund", "Process a refund for an order.", {"order_id": str, "amount": float})
async def sdk_process_refund(args):                # the action the compliance gate guards
    return {"content": [{"type": "text",
            "text": f"refunded {args['order_id']} amount {args['amount']}"}]}

**This cell:** the **`PostToolUse` normalization hook**. After `get_event` runs, it reads the
raw output, normalizes it, and returns the clean view as `additionalContext`. A `PostToolUse` hook
cannot rewrite a result that already ran, so it **appends** the cleaned companion the model then
reasons over.

In [ ]:
# ===== PostToolUse: append a normalized view of the raw output =====
def _output_text(tool_output):                     # pull text out of whatever shape the output has
    if isinstance(tool_output, str):  return tool_output
    if isinstance(tool_output, dict): tool_output = tool_output.get("content", tool_output)
    if isinstance(tool_output, list):                                   # list of content blocks
        return "".join(b.get("text", "") for b in tool_output if isinstance(b, dict))
    return json.dumps(tool_output) if tool_output is not None else ""

async def normalize_output(input_data, tool_use_id, context):   # runs AFTER a tool
    if input_data["tool_name"].split("__")[-1] != "get_event":  #   only clean get_event output
        return {}                                               #   otherwise observe and continue
    try:                                                        #   parse the raw payload safely
        raw = json.loads(_output_text(input_data.get("tool_output")))
    except Exception:                                           #   unparseable? do nothing
        return {}
    clean = normalize(raw)                                      #   apply the transform
    print("  [hook] appended normalized view:", clean)          #   show it firing
    return {"hookSpecificOutput": {"hookEventName": "PostToolUse",   # append, do not overwrite
            "additionalContext": "normalized: " + json.dumps(clean)}}

**This cell:** the **`PreToolUse` compliance gate**. Before `process_refund` runs, it checks
the amount against the limit and returns `deny` if it is over. This intercepts an invalid action
before it can happen, which is the only place you can truly stop it.

In [ ]:
# ===== PreToolUse: block a refund over the compliance limit =====
async def enforce_limit(input_data, tool_use_id, context):   # runs BEFORE a tool
    if input_data["tool_name"].split("__")[-1] != "process_refund":   # only guard refunds
        return {}                                            #   otherwise allow
    amount = input_data.get("tool_input", {}).get("amount", 0)   # the refund amount
    ok, reason = check_compliance(amount)                    #   apply the same rule as offline
    if not ok:                                               #   over the limit?
        print("  [gate]", reason)                            #     show the block
        return {"hookSpecificOutput": {"hookEventName": "PreToolUse",   # -> DENY
                "permissionDecision": "deny", "permissionDecisionReason": reason}}
    return {}                                                #   within limit -> allow

**This cell:** bundles the tools and wires both hooks with `HookMatcher`: the compliance gate
on `PreToolUse`, the normalizer on `PostToolUse`. This one options object carries both the
interception and the output transform.

In [ ]:
# ===== the hooked options =====
ops = create_sdk_mcp_server(name="ops", version="1.0.0",     # bundle the two tools
                            tools=[sdk_get_event, sdk_process_refund])
HOOKED = ClaudeAgentOptions(                                  # options with both hooks
    model=MODEL, mcp_servers={"ops": ops},
    allowed_tools=["mcp__ops__get_event", "mcp__ops__process_refund"],
    hooks={"PreToolUse":  [HookMatcher(hooks=[enforce_limit])],      # block invalid actions
           "PostToolUse": [HookMatcher(hooks=[normalize_output])]})  # normalize output
print("hooked workflow ready")

**This cell:** runs a request that calls `get_event`. Watch the `[hook]` line: after the raw
payload comes back, the PostToolUse hook appends the normalized view (ISO time, readable status)
for the model to use.

In [ ]:
# ===== run: raw output gets a normalized companion =====
if RUN_LIVE:                                      # needs a real key (and Node.js 18+)
    run_async(lambda: stream_run(HOOKED, "Get the latest event for order A1 and tell me its status and time."))
else:
    print("[skipped] expected: get_event returns raw fields, then the [hook] appends")
    print("          {status: shipped, event_time: 2024-...} for the model to read.")

**This cell:** runs a **small refund** ($50), under the limit. The compliance gate lets it
through, so the refund proceeds.

In [ ]:
# ===== run: a refund within the limit -> allowed =====
if RUN_LIVE:                                      # needs a real key
    run_async(lambda: stream_run(HOOKED, "Process a $50 refund for order A1."))
else:
    print("[skipped] expected: $50 is within the $500 limit, so the refund proceeds.")

**This cell:** runs a **large refund** ($900), over the limit. The compliance gate prints its
block and denies the call, so the over-limit refund never runs. Same tool, stopped by the rule.

In [ ]:
# ===== run: a refund over the limit -> blocked =====
if RUN_LIVE:                                      # needs a real key
    run_async(lambda: stream_run(HOOKED, "Process a $900 refund for order A1."))
else:
    print("[skipped] expected: the [gate] blocks $900 as over the $500 limit; needs a human.")

| anti-pattern | what to do instead |
|---|---|
| let the model reason over raw Unix time and codes | normalize at the hook layer into ISO and words |
| try to "undo" a bad action in `PostToolUse` | block it in `PreToolUse`; PostToolUse cannot undo |
| put a spending limit only in the prompt | enforce the threshold in a `PreToolUse` deny |
| hardcode the same rule in two places | share one `check_compliance` between offline and the hook |

**Lesson:** hooks are where you make output clean and actions safe. A `PostToolUse` hook
appends a normalized view so every later step reasons over consistent data, and a `PreToolUse` gate
enforces compliance rules like a spending threshold before the action can run. Prevention lives in
`PreToolUse`; `PostToolUse` shapes and annotates but cannot undo.

---

## Recap - normalize and enforce

| Hook | In this lab | Course topic |
|---|---|---|
| PostToolUse | append normalized output (Unix to ISO, code to word) | transform tool output before the model |
| PreToolUse | deny a refund over the limit | intercept and block invalid actions |
| Shared rule | one `check_compliance`, used offline and in the hook | compliance thresholds |

One principle to carry forward: **clean the data on the way out, and stop the bad action on the way
in.** To run live, paste a real key into **Setup 2/3** and re-run from the top. Then try it: lower
`REFUND_LIMIT` to 25 and watch the $50 refund get blocked too.